RAG Application using Typesense

In [2]:
from dotenv import load_dotenv
import typesense
import os
load_dotenv()

typesense_admin_key = os.getenv("TYPESENSE_ADMIN_KEY")
typesense_host = os.getenv("TYPESENSE_HOST")
typesense_port = os.getenv("TYPESENSE_PORT")

client = typesense.Client({
    'nodes': [{
        'host': typesense_host,
        'port': typesense_port,
        'protocol': 'https',
    }],
    'api_key': typesense_admin_key,
    'connection_timeout_seconds': 2
})

In [3]:
books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

In [4]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [6]:
search_parameters = {
    'q' : 'harry potter',
    'query_by' : 'title,authors',
    'filter_by' : 'publication_year: < 1998',
    'sort_by' : 'publication_year:desc',
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from google import genai

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [13]:
loader = TextLoader('data/text_files/sample1.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

c:\Users\Asus\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3508.55it/s]


In [15]:
docsearch = Typesense.from_documents(
    docs, embeddings, 
    typesense_client_params={
        'host':'7m2kphx6lasyw84rp-1.a1.typesense.net',
        'port':'443',
        'protocol':'https',
        'typesense_api_key': os.getenv("TYPESENSE_ADMIN_KEY"),
        'typesense_collection_name': 'lang-chain',
    }, 
)


In [19]:
q = "what is ASI?"
found_docs = docsearch.similarity_search(q, k=2)

print(found_docs[0].page_content)

Core Classifications of AI:
AI is primarily categorized by its capabilities and developmental stages:
Narrow AI (Weak AI): Systems built to excel at a singular, specific task. This is the only type of AI that currently exists.
General AI (AGI): A theoretical form of AI matching human-level adaptability and intellect across multiple domains.
Super AI (ASI): A hypothetical future intelligence that vastly surpasses all human cognitive capabilities.
